# 03 SRC Theory and Feature Architecture

**Scientific question.** How does an explicit physical state become a stable predictor?

**Learning objectives.** Derive and verify delayed angular and oscillatory state calculations.

**Prerequisites.** Install the project requirements. Configure the data directory for raw-record checks. Portable evidence review also uses the existing run artifacts.

**Dataset and run.** Course gearbox at 3.2 kHz, the project’s designated training and held-out development records, and the explicit diagnostic or blind availability files where relevant.

**Execution.** This clean-kernel replay reads the evidence cited in each code cell. Use the explicit project pipeline for full fitting and the notebook build script for review replay. The companion guide contains notation, figures, and worked examples.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
from IPython.display import Image, display

hint = os.environ.get("SRC_PROJECT_DIR")
locations = [Path(hint).resolve()] if hint else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in locations if (p / "src" / "src_gearbox").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open within the project or set SRC_PROJECT_DIR.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
DATA = Path(os.environ.get("SRC_DATA_DIR", ROOT / "data" / "raw"))
RESULTS = ROOT / "results"
START = 1792
N = 24000
pd.options.display.max_colwidth = 120
pd.options.display.width = 115
plt.rcParams.update({"figure.dpi": 105, "axes.grid": True, "grid.alpha": .18})

def table(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return None
    return pd.read_csv(file)

def evidence(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return {}
    return json.loads(file.read_text(encoding="utf-8"))

def show_fig(relative, width=780):
    file = ROOT / relative
    if file.is_file():
        display(Image(filename=str(file), width=width))
    else:
        print(f"NOT AVAILABLE YET · {relative}")

def forecast(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return {}
    with np.load(file, allow_pickle=False) as source:
        arrays = {key: source[key] for key in source.files}
    np.testing.assert_array_equal(arrays["target_indices"], np.arange(START, N))
    assert np.isfinite(arrays["actual"]).all()
    return arrays

def align_forecasts(reference, *others):
    for other in others:
        for key in ("target_indices", "actual", "rpm", "time_s"):
            np.testing.assert_array_equal(reference[key], other[key])

def draw_forecast(bundle, keys, view=slice(5000, 5200)):
    if not bundle:
        return
    y, t = bundle["actual"], bundle["time_s"]
    fig, ax = plt.subplots(2, 1, figsize=(9.0, 4.0), sharex=True, layout="constrained")
    ax[0].plot(t[view], y[view], color="#141a26", lw=1.2, label="Measured")
    for key, label in keys.items():
        if key in bundle:
            pred = bundle[key]
            ax[0].plot(t[view], pred[view], lw=0.9, label=label)
            ax[1].plot(t[view], (y-pred)[view], lw=0.9, label=label)
    ax[0].legend(ncol=3, fontsize=8)
    ax[0].set_ylabel("Std. vibration")
    ax[1].set(ylabel="Std. residual", xlabel="Acquisition time (s)")
    plt.show()

print("Read-only evidence replay.", f"Python {platform.python_version()}",
      f"NumPy {np.__version__}. One-step observed history through t−1.")

Read-only evidence replay. Python 3.12.14 NumPy 2.5.5. One-step observed history through t−1.


### From a lag row to structured model state

A delay vector $[x_{t-1},x_{t-3},x_{t-8}]$ is a feature map. Add a
past RPM basis, angle-sampled history, oscillator quadratures and selected
products. A grouped ridge model solves a scaled positive-definite system
with an unpenalized intercept. Grouping limits width: $p(p+1)/2$ quadratic
terms expand quickly even before any cubic terms. The regression is linear
in fitted weights and nonlinear in measured histories.

A finite oscillator demodulates the past with $\exp(-i2\pi k u_j)$ and rotates
to the target-clock phase using already observed RPM. For $r=\exp(-1/\tau)$,
the delayed state obeys $D_{t+1}=rD_t+d_t-r^M d_{t-M}$. Normalized quadratures
multiply by $2/\sum_{j=0}^{M-1}r^j$. The finite subtraction defines the
exact history. Companion Chapters 7–12 derive these maps and units.

In [2]:
from src_gearbox.src_model import angular_lags, oscillator_states, phase_turns, solve_readout

fs = 3200.0
rpm = np.linspace(125., 165., 760)
turns, target_turns = phase_turns(rpm, fs)
order, tau, memory = 148., 22., 96
x = 0.8*np.cos(2*np.pi*order*turns)
state = oscillator_states(x, fs, rpm, order=order, tau=tau, memory=memory)
weights = np.exp(-np.arange(memory)/tau)
idx = np.arange(memory, len(x))
prior = idx[:, None] - np.arange(1, memory+1)
rotation = np.exp(2j*np.pi*order*(target_turns[idx, None]-turns[prior]))
direct = 2*np.sum(weights*x[prior]*rotation, axis=1)/weights.sum()
np.testing.assert_allclose(state[idx], direct, rtol=2e-11, atol=2e-11)
mutated = x.copy()
mutated[420:] += 50
mutated_rpm = rpm.copy()
mutated_rpm[420:] += 5
probe = oscillator_states(mutated, fs, mutated_rpm, order=order, tau=tau, memory=memory)
np.testing.assert_array_equal(state[:421], probe[:421])
print("Finite oscillator direct agreement =", float(np.max(np.abs(state[idx]-direct))))

Finite oscillator direct agreement = 4.69137544346138e-14


In [3]:
t = np.arange(1800)/fs
affine_x = 2*t+1
fixed_rpm = np.full(len(t), 120.0)
wanted = np.array([1690,1730])
observed = angular_lags(affine_x, fixed_rpm, fs, wanted, steps=4, cycles=1)
shifts = np.arange(1,5)/(37*4)*fs/2
truth = 2*((wanted[:,None]-shifts)/fs)+1
np.testing.assert_allclose(observed, truth, atol=2e-7, rtol=0)
gram_example = np.array([[0.,1.],[1.,1.],[2.,1.],[3.,1.]])
desired = np.array([.2,1.,1.7,3.])
penalty = np.array([.25,0.])
coef, diag = solve_readout(gram_example, desired, penalty)
reference = np.linalg.solve(gram_example.T@gram_example + len(desired)*np.diag(penalty),
                            gram_example.T@desired)
np.testing.assert_allclose(coef, reference)
display({"slope":coef[0],"unpenalized_intercept":coef[1],**diag})

{'slope': np.float64(0.7583333333333331),
 'unpenalized_intercept': np.float64(0.3375000000000005),
 'backend': 'cholesky',
 'gram_condition_estimate': 18.374999999999993,
 'iterations': 1}

### Interpretation and limits

The primitive checks isolate unit, phase and dependency errors without
using rig data. A stable recurrence does not make the model a trained
recurrent network. The oscillator settings are chosen in the guarded
outer search, while the coefficients are fitted by the inner solver.

Read the [Companion Guide](../docs/companion_guide.md) and the [Project README](../README.md).